# 01 · Veri ve baseline: İzmir Adnan Menderes (LTBJ)

**Soru:** GFS'in bir gün önceden yaptığı rüzgâr tahmini havalimanında ne kadar tutuyor, basit bir düzeltmeyle ne kadar iyileşir?

İki veri kaynağımız var. İlki METAR: Adnan Menderes'in gerçek ölçüm verileri. Saatte 2 kez ölçüm yapar, x.20 ve x.50 dakikalarda. İkincisi ABD hava modeli GFS'in tahminleri, ama geniş bir bölge (~25 km) için tahminde bulunuyor.

| | Gerçek (`metar`) | Tahmin (`gfs`) |
|---|---|---|
| Kaynak | Iowa State Mesonet (IEM) METAR arşivi | Open-Meteo Previous Runs API, `previous_day1` |
| Sıklık | Saatte 2 (:20, :50) | Saatte 1 (:00) |
| Birim | knot, 10 dk ortalama, tam sayıya yuvarlı | knot, anlık |

Aralık 2024-03-01 → 2026-11-02, saatler UTC (TR = UTC+3). Veriyi indirmek için: `uv run python -m ruzgar.download`

## 1. Veriyi yükle

In [1]:
import pandas as pd
metar = pd.read_parquet("../data/raw/metar_ltbj.parquet")
gfs = pd.read_parquet("../data/raw/gfs_ltbj.parquet")

## 2. Veriye bak

In [2]:
display(metar.head(10))
display(gfs.head(10))

,station,time,speed_kn,dir_deg,gust_kn
0,LTBJ,2024-03-01 00:20:00+00:00,1.0,NaN,NaN
1,LTBJ,2024-03-01 00:50:00+00:00,2.0,NaN,NaN
2,LTBJ,2024-03-01 01:20:00+00:00,4.0,220.0,NaN
3,LTBJ,2024-03-01 01:50:00+00:00,5.0,150.0,NaN
4,LTBJ,2024-03-01 02:20:00+00:00,4.0,140.0,NaN
5,LTBJ,2024-03-01 02:50:00+00:00,2.0,160.0,NaN
6,LTBJ,2024-03-01 03:20:00+00:00,2.0,NaN,NaN
7,LTBJ,2024-03-01 03:50:00+00:00,6.0,350.0,NaN
8,LTBJ,2024-03-01 04:20:00+00:00,3.0,360.0,NaN
9,LTBJ,2024-03-01 04:50:00+00:00,4.0,350.0,NaN


,time,speed_kn,dir_deg,gust_kn
0,2024-03-01 00:00:00+00:00,1.7,159,1.9
1,2024-03-01 01:00:00+00:00,1.8,139,2.1
2,2024-03-01 02:00:00+00:00,2.0,287,1.6
3,2024-03-01 03:00:00+00:00,1.6,346,1.7
4,2024-03-01 04:00:00+00:00,0.4,297,0.4
5,2024-03-01 05:00:00+00:00,0.9,207,0.6
6,2024-03-01 06:00:00+00:00,2.6,117,3.7
7,2024-03-01 07:00:00+00:00,2.6,107,2.7
8,2024-03-01 08:00:00+00:00,4.1,121,4.5
9,2024-03-01 09:00:00+00:00,4.4,165,5.2


In [3]:
display(metar.tail(5))
display(gfs.tail(5))

,station,time,speed_kn,dir_deg,gust_kn
48279,LTBJ,2026-11-02 21:50:00+00:00,3.0,160.0,NaN
48280,LTBJ,2026-11-02 22:20:00+00:00,2.0,NaN,NaN
48281,LTBJ,2026-11-02 22:50:00+00:00,2.0,140.0,NaN
48282,LTBJ,2026-11-02 23:20:00+00:00,2.0,NaN,NaN
48283,LTBJ,2026-11-02 23:50:00+00:00,3.0,130.0,NaN


,time,speed_kn,dir_deg,gust_kn
23443,2026-11-02 19:00:00+00:00,3.3,135,4.1
23444,2026-11-02 20:00:00+00:00,3.7,137,4.7
23445,2026-11-02 21:00:00+00:00,3.3,140,3.9
23446,2026-11-02 22:00:00+00:00,3.3,135,3.5
23447,2026-11-02 23:00:00+00:00,3.3,126,3.9


## 3. Hizalama

GFS saat başında, METAR :20 ve :50'de. Her GFS saatine **en yakın** METAR ölçümünü eşliyoruz (±15 dk). 20 dakika içinde durumlar ani şekilde değişebilir, bu yüzden en yakını almak daha mantıklı. Diğer seçenek (iki ölçümün ortalaması) sonra ölçülerek karşılaştırılacak.

In [4]:
aligned = pd.merge_asof(
    gfs, metar,
    on="time",
    direction="nearest",
    tolerance=pd.Timedelta("15min"),
    suffixes=("_gfs", "_obs"),
)
aligned = aligned.dropna(subset=["speed_kn_obs"])
aligned.head()

,time,speed_kn_gfs,dir_deg_gfs,gust_kn_gfs,station,speed_kn_obs,dir_deg_obs,gust_kn_obs
1,2024-03-01 01:00:00+00:00,1.8,139,2.1,LTBJ,2.0,NaN,NaN
2,2024-03-01 02:00:00+00:00,2.0,287,1.6,LTBJ,5.0,150.0,NaN
3,2024-03-01 03:00:00+00:00,1.6,346,1.7,LTBJ,2.0,160.0,NaN
4,2024-03-01 04:00:00+00:00,0.4,297,0.4,LTBJ,6.0,350.0,NaN
5,2024-03-01 05:00:00+00:00,0.9,207,0.6,LTBJ,4.0,350.0,NaN


## 4. Baseline: ham GFS'in hatası

MAE (Mean Absolute Error): tahmin ortalamada kaç knot şaşıyor. İşarete bakmaz, yalnız sapmanın büyüklüğünü ölçer.
→ “GFS ortalama 4,5 knot yanılıyor.”

Bias: tahmin hangi yöne şaşıyor. İşaretli ortalamadır, artılar ve eksiler birbirini götürür.
→ “GFS ortalama 3,7 knot düşük tahmin ediyor.” Eksi işaret düşük tahmin demek.

In [5]:
aligned["hata"] = aligned["speed_kn_gfs"] - aligned["speed_kn_obs"]
aligned[["time", "speed_kn_gfs", "speed_kn_obs", "hata"]].head()

,time,speed_kn_gfs,speed_kn_obs,hata
1,2024-03-01 01:00:00+00:00,1.8,2.0,-0.2
2,2024-03-01 02:00:00+00:00,2.0,5.0,-3.0
3,2024-03-01 03:00:00+00:00,1.6,2.0,-0.4
4,2024-03-01 04:00:00+00:00,0.4,6.0,-5.6
5,2024-03-01 05:00:00+00:00,0.9,4.0,-3.1


In [6]:
mae = aligned["hata"].abs().mean()
bias = aligned["hata"].mean()
print(f"MAE: {mae:.2f} knot   Bias: {bias:.2f} knot")

MAE: 4.47 knot   Bias: -3.72 knot


### Saate göre

In [7]:
saatlik = aligned.groupby(aligned["time"].dt.hour)["hata"].agg(
    mae=lambda h: h.abs().mean(),
    bias=lambda h: h.mean(),
)
saatlik.round(2)

,mae,bias
time,,
0,3.87,-3.02
1,3.95,-3.11
2,3.99,-3.20
3,4.05,-3.31
4,4.29,-3.59
5,4.49,-3.77
6,4.72,-3.99
7,4.80,-4.09
8,4.74,-3.90


### Aya göre

In [8]:
aylık = aligned.groupby(aligned["time"].dt.month)["hata"].agg(
    mae=lambda h: h.abs().mean(),
    bias=lambda h: h.mean(),
)
aylık.round(2)

,mae,bias
time,,
1,3.05,-1.26
2,4.18,-3.02
3,3.88,-2.80
4,4.10,-3.12
5,3.72,-3.03
6,5.43,-5.02
7,5.51,-5.20
8,6.06,-5.84
9,5.34,-5.00


### Yalnız yaz (Haziran–Eylül)

In [9]:
yaz = aligned[aligned["time"].dt.month.isin([6, 7, 8, 9])]
yaz_saatlik = yaz.groupby(yaz["time"].dt.hour)["hata"].agg(
    mae=lambda h: h.abs().mean(),
    bias="mean",
)
yaz_saatlik.round(2)

,mae,bias
time,,
0,5.21,-4.80
1,5.50,-5.08
2,5.44,-5.01
3,5.63,-5.17
4,5.98,-5.64
5,6.56,-6.20
6,6.71,-6.47
7,6.44,-6.33
8,6.01,-5.75


**Gözlemler**
- GFS sistematik olarak düşük tahmin ediyor (bias ≈ −3,7 kn); hatanın büyük kısmı kayma, yani düzeltilebilir.
- Yazın bias −5 ile −5,8 kn (Ağustos en kötü), kışın −1,3 ile −3.
- Yaz içinde en büyük hata TR 08–10 ve 21–23, en küçük TR 17–18. Hipotez: GFS imbatın gücünü değil başlangıç ve bitiş zamanını kaçırıyor.

## 5. Dürüst test: zamana göre bölme

2025 kasım öncesi eğitim sonrasını test seti olarak alacağız. Bias'ı tüm veriden hesaplayıp aynı veride ölçmek leakage olur. Test tam bir yıl, böylece bütün mevsimler içinde.

In [10]:
kesim = pd.Timestamp("2025-11-01", tz="UTC")
train = aligned[aligned["time"] < kesim]
test = aligned[aligned["time"] >= kesim]

print(len(train), len(test))

14629 8804


## 6. Bias düzeltmesi

### Sabit düzeltme

In [11]:
# 1) Düzeltmeyi YALNIZ train'den öğren
duzeltme = train["hata"].mean()

# 2) Test'te uygula
test_tahmin = test["speed_kn_gfs"] - duzeltme

# 3) Test'te iki MAE'yi karşılaştır
mae_ham = test["hata"].abs().mean()
mae_duz = (test_tahmin - test["speed_kn_obs"]).abs().mean()
print(f"Ham GFS MAE: {mae_ham:.2f}   Sabit düzeltmeli MAE: {mae_duz:.2f}")

Ham GFS MAE: 4.41   Sabit düzeltmeli MAE: 3.80


### Aylık düzeltme

In [12]:
# 1) Train'den her ayın bias'ını öğren → 12 sayılık tablo
aylik_duz = train.groupby(train["time"].dt.month)["hata"].mean()
print(aylik_duz.round(2))

# 2) Test'teki her satıra kendi ayının düzeltmesini getir
test_ay = test["time"].dt.month
test_duz = test_ay.map(aylik_duz)

# 3) Uygula ve ölç
test_tahmin_aylik = test["speed_kn_gfs"] - test_duz
mae_aylik = (test_tahmin_aylik - test["speed_kn_obs"]).abs().mean()
print(f"Ham: {mae_ham:.2f}   Sabit: {mae_duz:.2f}   Aylık: {mae_aylik:.2f}")

time
1    -1.62
2    -4.44
3    -1.82
4    -3.10
5    -3.16
6    -5.40
7    -5.00
8    -5.55
9    -4.56
10   -3.49
11   -3.38
12   -2.73
Name: hata, dtype: float64
Ham: 4.41   Sabit: 3.80   Aylık: 3.65


## Sonuç

Ham: 4.41   Sabit: 3.80   Aylık: 3.65

Model eğitimine geçmeden matematiksel olarak hatayı düşürmeyi başardık. Veri sayısı çok olmadığı için overfittingi engellemek için saat bazlı eğitim yapmıyoruz.